# 03. Análisis de KPIs de PlayLoop

**Objetivo:** analizar la retención, la monetización y el rendimiento
de los canales de adquisición para evaluar CoinRush.

**Fechas corregidas:** interpretadas desde los originales con formatos explícitos.
**Pendiente:** validar los tres gastos superiores a 1.000 € antes de interpretar costes por canal.


In [12]:
# Importamos librerias
import pandas as pd
from pathlib import Path
from dotenv import dotenv_values
from sqlalchemy import URL, create_engine, text

In [13]:
# Localizamos la raiz
# Esto funciona si el notebook se ejecuta desde la raíz o desde notebooks.
carpeta_actual = Path.cwd()
raiz_proyecto = (
    carpeta_actual.parent
    if carpeta_actual.name == 'notebooks'
    else carpeta_actual
)

# Leemos credenciales sin mostrarlas 
configuracion =  dotenv_values(raiz_proyecto / '.env')

# Preparamos la dirección de conexión a PostgreSQL.
url = URL.create(
    'postgresql+psycopg2',
    username=configuracion['DB_USER'],
    password=configuracion['DB_PASSWORD'],
    host=configuracion['DB_HOST'],
    port=int(configuracion['DB_PORT']),
    database=configuracion['DB_NAME']
)

# Creamos el gestor de conexiones.
engine = create_engine(url, connect_args={'connect_timeout': 5})

In [14]:
# Contamos los usuarios almacenados en PostgreSQL.
consulta = text("""
    SELECT COUNT(*) AS total_usuarios
    FROM public.usuarios
""")

# Ejecutamos la consulta y guardamos el resultado en un DataFrame.
with engine.connect() as conexion:
    resultado = pd.read_sql(consulta, conexion)

# Mostramos la tabla resultante en el notebook.
resultado

,total_usuarios
0,8000


**Resumen — base de usuarios**

- **Qué es:** total de jugadores únicos de la tabla de usuarios.
- **Resultado:** 8.000 usuarios.
- **Uso:** referencia para los porcentajes sobre toda la base; cada KPI puede usar otro denominador.


#### DAU - Usuarios activos diarios
¿Cuántos jugadores distintos abren el juego cada día?
Consideramos activo a un jugador si tiene al menos una sesión ese día.
- **DAU medio**: promedio de jugadores únicos por día sobre todos los días de actividad

In [15]:
# Calculamos los usuarios únicos de cada día y después su promedio.
consulta_dau = text("""
    -- Creamos un resultado temporal con una fila por fecha.
    WITH activos_por_dia AS (
        SELECT
            fecha,
            COUNT(DISTINCT user_id) AS activos
        FROM public.sesiones
        GROUP BY fecha
    )

    -- Calculamos el promedio de los recuentos diarios.
    SELECT
        ROUND(AVG(activos), 0) AS dau_medio,
        COUNT(*) AS dias_con_actividad
    FROM activos_por_dia
""")

# Ejecutamos la consulta y traemos el resultado a pandas.
with engine.connect() as conexion:
    resumen_dau = pd.read_sql(consulta_dau, conexion)

resumen_dau

,dau_medio,dias_con_actividad
0,601.0,56


**Resumen — DAU medio**

- **DAU:** jugadores distintos con sesión en una fecha; varias sesiones del mismo jugador cuentan una vez.
- **DAU medio:** promedio de esos recuentos en los días con actividad.
- **Resultado:** 601 jugadores diarios de media (601,48 sin redondear), en 56 días.
- **Periodo:** 28/05/2024–22/07/2024. No se añaden ceros para fechas sin actividad.
- **Diferencia con el ejercicio:** sin excluir sesiones por duración da 602; mantenemos nuestro filtro de limpieza.


#### MAU y stickiness 
- **MAU**: jugadores únicos con sesiones en los 30 días previos a la extracción del 22/07/2024
- **Stickiness:** DAU medio / MAU × 100.


In [18]:
consulta_mau = text("""
    WITH mau AS(
-- MAU: jugadores unicos en los 30 dias previos a la extraccion
       SELECT COUNT(DISTINCT user_id) AS mau
        FROM public.sesiones
        WHERE fecha > DATE '2024-07-22' - 30
    ),

    dau AS (
        -- Promedio de usuarios activos de todos los días con actividad.
        SELECT AVG(activos) AS dau
        FROM (
            SELECT
                fecha,
                COUNT(DISTINCT user_id) AS activos
            FROM public.sesiones
            GROUP BY fecha
        ) AS activos_por_dia
    )

    SELECT
        mau.mau,
        ROUND(dau.dau, 0) AS dau_medio,
        ROUND(100.0 * dau.dau / NULLIF(mau.mau, 0), 1) AS stickiness_pct
    FROM mau, dau
""")

# Ejecutamos SQL y recogemos el resultado en pandas.
with engine.connect() as conexion:
    resumen_actividad = pd.read_sql(consulta_mau, conexion)

resumen_actividad

,mau,dau_medio,stickiness_pct
0,7398,601.0,8.1


**Resumen — MAU y stickiness**

- **MAU:** jugadores distintos activos del 23/06 al 22/07/2024: **7.398**.
- **DAU medio:** **601**, calculado sobre todo el periodo del dataset.
- **Stickiness:** DAU medio / MAU × 100 = **8,1%**, usando el DAU sin redondear.
- **Alcance:** seguimos el ejercicio, con ventanas distintas para DAU y MAU. No equivale estrictamente al promedio diario dentro de la ventana del MAU ni significa que el mismo 8,1% entre todos los días.


### Retención D1 — ¿vuelven al día siguiente?
- **Cohorte:** jugadores que ya pudieron cumplir su día 1 antes o en la
  fecha de extracción, el 22 de julio de 2024.
- **Retenidos:** jugadores de esa cohorte con una sesión en instalación + 1 día.
- **Retención D1:** retenidos / cohorte × 100.

Cada jugador cuenta una sola vez, aunque tenga varias sesiones ese día.

In [22]:
# Calculamos la retención D1
consulta_d1 = text("""
    WITH cohorte AS (
    -- Solo los que llegaron a D1
    SELECT user_id
    FROM public.usuarios
    WHERE DATE '2024-07-22' - fecha_instalacion >= 1
    ),

    retenidos AS (
    SELECT DISTINCT s.user_id 
    FROM public.sesiones AS s
    JOIN public.usuarios AS u
        ON s.user_id = u.user_id
    WHERE s.fecha - u.fecha_instalacion = 1
    )

    SELECT 
        COUNT(DISTINCT c.user_id) AS cohorte,
        COUNT(DISTINCT r.user_id) AS retenidos,
        -- Dividimos los retenidos entre los jugadores elegibles
        ROUND(
            100.0*COUNT(DISTINCT r.user_id)
            / NULLIF(COUNT(DISTINCT c.user_id),0),
            1
            ) AS retencion_d1_pct
    
    FROM cohorte AS c
    -- Conservamos también a quienes no volvieron para incluirlos en el total
    LEFT JOIN retenidos AS r
        ON c.user_id = r.user_id
""")

# Ejecutamos la consulta y guardamos su resultado en un DataFrame
with engine.connect() as conexion:
    retencion_d1 = pd.read_sql(consulta_d1,conexion)

retencion_d1

,cohorte,retenidos,retencion_d1_pct
0,8000,3261,40.8


**Resumen — retención D1**

- **Cohorte:** jugadores que pudieron cumplir D1 al corte del 22/07: **8.000**.
- **Retenidos:** tuvieron una sesión exactamente al día siguiente de instalar: **3.261**.
- **Fórmula:** retenidos / cohorte × 100 = **40,8%**.
- **Clave:** ser elegible no significa haber vuelto; varias sesiones en D1 cuentan una sola vez.


#### Retención D1, D7 y D30
¿Qué porcentaje de jugadores vuelve exactamente al día siguiente, a los siete días y a los treinta días de instalar?

Usamos el 22 de julio de 2024 como fecha de extracción. Para cada plazo calculamos:
- **Cohorte:** jugadores que ya tuvieron tiempo de alcanzar ese día.
- **Retenidos:** jugadores de esa cohorte con al menos una sesión
  exactamente en instalación + N días.
- **Retención:** retenidos / cohorte × 100.


In [33]:
# Calculamos la retención en los días 1, 7 y 30.
consulta_retencion = text("""
    -- Creamos los tres plazos que queremos evaluar.
    WITH dias(n) AS (VALUES (1), (7), (30))

    SELECT
        d.n AS dia,
        COUNT(DISTINCT u.user_id) AS cohorte,
        COUNT(DISTINCT s.user_id) AS retenidos,

        -- Porcentaje de jugadores elegibles que volvieron ese día.
        ROUND(
            100.0 * COUNT(DISTINCT s.user_id)
            / NULLIF(COUNT(DISTINCT u.user_id), 0),
            1
        ) AS retencion_pct

    FROM dias AS d

    -- Incluimos a quienes ya tuvieron tiempo de alcanzar cada plazo.
    JOIN public.usuarios AS u
        ON DATE '2024-07-22' - u.fecha_instalacion >= d.n

    -- Buscamos actividad exactamente N días DESPUÉS de instalar.
    LEFT JOIN public.sesiones AS s
        ON s.user_id = u.user_id
        AND s.fecha - u.fecha_instalacion = d.n

    GROUP BY d.n
    ORDER BY d.n
""")

# Ejecutamos la consulta corregida y actualizamos el DataFrame.
with engine.connect() as conexion:
    retencion = pd.read_sql(consulta_retencion, conexion)

retencion
retencion

,dia,cohorte,retenidos,retencion_pct
0,1,8000,3261,40.8
1,7,6808,1005,14.8
2,30,2470,176,7.1


**Resumen — retención D1 / D7 / D30**

| Plazo | Cohorte elegible | Retenidos | Retención |
|---|---:|---:|---:|
| D1 | 8.000 | 3.261 | 40,8% |
| D7 | 6.808 | 1.005 | 14,8% |
| D30 | 2.470 | 176 | 7,1% |

- **Qué mide:** volver exactamente N días después de instalar; no en cualquier momento de los primeros N días.
- **Cohorte madura:** solo jugadores con al menos N días observables al 22/07/2024.
- **Lectura:** cada plazo tiene su propio denominador; no seguimos necesariamente al mismo grupo en los tres puntos.
- **Comprobación:** D1 coincide con la consulta individual. El desglose por canal sigue pendiente.


#### Embudo de progresión - niveles 1,3,6,10
¿Cuántos jugadores llegan a jugar cada nivel clave?

Contamos jugadores únicos con al menos una partida registrada en los niveles 1, 3, 6 y 10, y calculamos su porcentaje sobre el total de usuarios.
- Cada jugador cuenta 1 vez por nivel
- Tener una partida en un nivel indica que lo jugó no que lo superó

In [26]:
# Contamos jugadores únicos en los niveles clave.
consulta_embudo = text("""
    SELECT 
        nivel,
        COUNT(DISTINCT user_id) AS jugadores,
        -- Dividimos entre el total de usuarios
        ROUND(
            100.0*COUNT(DISTINCT user_id)
            / NULLIF((SELECT COUNT(*) FROM public.usuarios),0),
            1
            ) AS pct_del_total
    FROM public.partidas
    WHERE nivel IN (1,3,6,10)
    GROUP BY nivel
    ORDER BY nivel
""")

# Ejecutamos la consulta y guardamos el resultado en pandas.
with engine.connect() as conexion:
    embudo = pd.read_sql(consulta_embudo, conexion)

embudo

,nivel,jugadores,pct_del_total
0,1,8000,100.0
1,3,6320,79.0
2,6,3680,46.0
3,10,960,12.0


**Resumen — embudo de niveles**

- **Qué cuenta:** jugadores con una partida en cada nivel exacto, sin repetir intentos del mismo jugador.
- **Resultados:** nivel 1: **8.000 (100%)**; nivel 3: **6.320 (79%)**; nivel 6: **3.680 (46%)**; nivel 10: **960 (12%)**.
- **Denominador:** los 8.000 usuarios, no los jugadores del nivel anterior.
- **Límite:** jugar un nivel no significa superarlo; la antigüedad de los jugadores no está ajustada.


#### El escalón entre los niveles 6 y 7
¿Qué porcentaje de quienes alcanzan el nivel 6 no llega al nivel 7?
La caída identifica un punto que investigar, pero no demuestra por sí sola un problema de diseño ni un abandono definitivo. 

In [35]:
# Contamos jugadores únicos en ambos niveles y calculamos la caída.
consulta_caida = text("""
    WITH niveles AS (
    SELECT 
        COUNT(
            DISTINCT CASE WHEN nivel >= 6 THEN user_id END
        ) AS llegan_al_6,
        COUNT(
            DISTINCT CASE WHEN nivel >= 7 THEN user_id END
        ) AS llegan_al_7
    FROM public.partidas

    )

    SELECT 
        llegan_al_6,
        llegan_al_7,
        ROUND(
        100.0 * (llegan_al_6 - llegan_al_7)
        / NULLIF(llegan_al_6,0),
        1
    ) AS caida_pct

    FROM niveles
""")

# Ejecutamos SQL y guardamos su respuesta en un DataFrame.
with engine.connect() as conexion:
    caida_nivel_6 = pd.read_sql(consulta_caida, conexion)

# Mostramos el resultado.
caida_nivel_6

,llegan_al_6,llegan_al_7,caida_pct
0,3680,1920,47.8


**Resumen — caída entre niveles 6 y 7**

- **Alcanzan al menos el nivel 6:** **3.680** jugadores (nivel >= 6).
- **Alcanzan al menos el nivel 7:** **1.920** jugadores (nivel >= 7).
- **Diferencia:** **1.760** jugadores; caída = (3.680 − 1.920) / 3.680 × 100 = **47,8%**.
- **Supuesto:** los niveles se recorren en orden. Aquí usamos umbrales, a diferencia del nivel exacto del embudo anterior.
- **Lectura:** punto para investigar intentos, dificultad y tiempo disponible; no prueba un problema de diseño ni abandono definitivo.


#### Sesiones por jugador y duración media
¿Con qué frecuencia abren el juego los jugadores activos y cuánto duran sus sesiones?
- **Sesiones por jugador:** total de sesiones / jugadores únicos presentes en la tabla de sesiones, durante todo el periodo.
- **Duración media:** promedio de duración de las sesiones entre 1 y 3,600 segundos, ambos incluidos.




In [36]:
# Calculamos sesiones por jugador activo y duración media de sesión.
consulta_sesiones = text("""
    SELECT 
        ROUND(
        COUNT(*)*1.0 / NULLIF(COUNT(DISTINCT user_id),0),
        2) AS sesiones_por_jugador,
    -- Duraciones entre 1 segundo y 1 hora
    ROUND(
        AVG(
            CASE        
                WHEN duracion_segundos BETWEEN 1 AND 3600
                THEN duracion_segundos
            END
        ),1   
    ) AS duracion_media_seg
    FROM public.sesiones
""")

# Ejecutamos la consulta y guardamos el resultado en pandas.
with engine.connect() as conexion:
    resumen_sesiones = pd.read_sql(consulta_sesiones, conexion)

resumen_sesiones

,sesiones_por_jugador,duracion_media_seg
0,5.57,183.6


**Resumen — sesiones y duración**

- **Sesiones por jugador activo:** total de sesiones / usuarios distintos con sesiones = **5,57** en todo el periodo; no son sesiones diarias.
- **Duración media por sesión:** **183,6 segundos**, aproximadamente **3,06 minutos**.
- **Filtro:** duraciones entre 1 y 3.600 segundos. La limpieza ya había excluido las sesiones fuera del rango.
- **Detalle SQL:** CASE filtra únicamente los valores de AVG; no filtra por sí mismo las filas de COUNT.


#### Actividad por día de la semana
**Hipótesis:** los jugadores juegan más los fines de semana.
Contamos, para cada día de la semana:
- **Jugadores activos:** jugadores distintos con al menos una sesión
  en ese día de la semana durante todo el periodo.
- **Sesiones:** total de sesiones registradas.

In [38]:
consulta_semana = text("""
    SELECT
        EXTRACT(ISODOW FROM fecha)::INTEGER AS dia_num,
        -- Asignamos nombre a día
        CASE EXTRACT(ISODOW FROM fecha)
            WHEN 1 THEN 'lunes'
            WHEN 2 THEN 'martes'
            WHEN 3 THEN 'miercoles'
            WHEN 4 THEN 'jueves'
            WHEN 5 THEN 'viernes'
            WHEN 6 THEN 'sabado'
            WHEN 7 THEN 'domingo'
        END AS dia,
        COUNT(DISTINCT user_id) AS jugadores_Activos,
        COUNT(*) AS sesiones

    FROM public.sesiones
    GROUP BY dia_num, dia
    ORDER BY dia_num
""")


# Ejecutamos la consulta y mostramos el resultado.
with engine.connect() as conexion:
    actividad_semana = pd.read_sql(consulta_semana, conexion)

actividad_semana


,dia_num,dia,jugadores_activos,sesiones
0,1,lunes,3941,6695
1,2,martes,3564,5924
2,3,miercoles,3719,6127
3,4,jueves,3748,6316
4,5,viernes,3782,6333
5,6,sabado,3898,6404
6,7,domingo,3990,6774


**Resumen — actividad por día de la semana**

- **Jugadores activos:** personas distintas por día de la semana acumulado; quien juega varios lunes cuenta una vez en lunes. No es el DAU medio de los lunes.
- **Sesiones:** todos los eventos de cada grupo, incluidos intentos repetidos del mismo jugador.
- **Resultados destacados:** domingo, **3.990 jugadores / 6.774 sesiones**; lunes, **3.941 / 6.695**; sábado, **3.898 / 6.404**.
- **Lectura:** domingo encabeza ambos recuentos, pero sábado no supera al lunes. No basta para afirmar que se juega mucho más todos los fines de semana ni para decidir un lanzamiento en sábado.
- **Comparación:** el periodo contiene ocho fechas de cada día de la semana. Comparar fin de semana y laborables requiere normalizar por fechas, porque son dos días frente a cinco.
- **Límite:** estos agregados no demuestran el efecto que tendría lanzar un evento en un día concreto.
